# Notebook 4: Metadata Filtering
**Scope k-NN search by episode title, date range, or any structured field**

Pure semantic search treats every chunk equally. But often you want:
- Only results from a **specific episode**
- Only results from episodes published **after a certain date**
- Combine structured filters with vector similarity

OpenSearch supports this via **post-filtering** on k-NN queries. The vector search
runs first, then filters are applied to the candidate set.

### Prerequisites
- Notebook 01 completed (podcasts indexed)

## 0 · Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from sentence_transformers import SentenceTransformer
from src.opensearch_client import (
    get_client, knn_search, knn_search_filtered, INDEX_NAME
)

client = get_client()
model = SentenceTransformer('all-MiniLM-L6-v2')

def embed(text: str) -> list[float]:
    return model.encode(text, normalize_embeddings=True).tolist()

print(f'✅ {client.count(index=INDEX_NAME)["count"]} chunks indexed')

✅ 1018 chunks indexed


## 1 · Discover Available Metadata

Before filtering, let's see what episode titles and date ranges exist in our index.

In [2]:
# List all episodes and their chunk counts
resp = client.search(
    index=INDEX_NAME,
    body={
        'size': 0,
        'aggs': {
            'episodes': {
                'terms': {'field': 'episode_title', 'size': 50, 'order': {'_key': 'asc'}}
            }
        }
    }
)

episodes = resp['aggregations']['episodes']['buckets']
print(f'Found {len(episodes)} episodes:\n')
for ep in episodes:
    print(f"  [{ep['doc_count']:3d} chunks]  {ep['key']}")

Found 33 episodes:

  [ 26 chunks]  Adding ML layer to Search: Hybrid Search Optimizer with Daniel Wrigley and Eric Pugh
  [ 33 chunks]  Amin Ahmad - CTO, Vectara - Algolia / Elasticsearch-like search product on neural search principles
  [ 47 chunks]  Atita Arora - Search Relevance Consultant - Revolutionizing E-commerce with Vector Search
  [ 17 chunks]  Berlin Buzzwords 2024 - Alessandro Benedetti - LLMs in Solr
  [ 14 chunks]  Berlin Buzzwords 2024 - Doug Turnbull - Learning in Public
  [ 10 chunks]  Berlin Buzzwords 2024 - Sonam Pankaj - EmbedAnything
  [ 47 chunks]  Bob van Luijt (CEO, Semi) on the Weaviate vector search engine
  [ 31 chunks]  Code search, Copilot, LLM prompting with empathy and Artifacts with John Berryman
  [ 32 chunks]  Connor Shorten - PhD Researcher - Florida Atlantic University & Founder at Henry AI Labs
  [ 48 chunks]  Connor Shorten - Research Scientist, Weaviate - ChatGPT, LLMs, Form vs Meaning
  [ 28 chunks]  Daniel Tunkelang - Leading Search Consultant

In [3]:
# What pub_date values do we have?
resp = client.search(
    index=INDEX_NAME,
    body={
        'size': 0,
        'aggs': {
            'dates': {
                'terms': {'field': 'pub_date', 'size': 50, 'order': {'_key': 'desc'}}
            }
        }
    }
)

print('Publication dates (most recent first):\n')
for d in resp['aggregations']['dates']['buckets'][:15]:
    print(f"  {d['key']}  ({d['doc_count']} chunks)")

Publication dates (most recent first):

  Wed, 26 Jun 2024 13:42:56 GMT  (21 chunks)
  Wed, 21 Dec 2022 20:35:43 GMT  (30 chunks)
  Wed, 19 Jan 2022 21:02:57 GMT  (26 chunks)
  Wed, 17 May 2023 08:12:12 GMT  (47 chunks)
  Wed, 16 Feb 2022 16:14:37 GMT  (33 chunks)
  Wed, 15 May 2024 12:57:55 GMT  (20 chunks)
  Wed, 01 May 2024 13:54:39 GMT  (29 chunks)
  Tue, 30 Aug 2022 07:27:26 GMT  (37 chunks)
  Tue, 12 Apr 2022 12:29:08 GMT  (40 chunks)
  Thu, 23 Dec 2021 16:01:59 GMT  (26 chunks)
  Thu, 23 Dec 2021 13:32:52 GMT  (32 chunks)
  Thu, 23 Dec 2021 13:28:43 GMT  (40 chunks)
  Thu, 23 Dec 2021 13:17:15 GMT  (47 chunks)
  Thu, 19 Sep 2024 11:02:40 GMT  (10 chunks)
  Thu, 18 Jul 2024 11:10:42 GMT  (14 chunks)


## 2 · Filter by Episode Title

Restrict vector search to chunks from a single episode.
Useful when a user says *"What did Yury Malkov say about..."*

In [4]:
query = "How does the HNSW algorithm handle high-dimensional data?"

# Unfiltered results from any episode
print('UNFILTERED k-NN search:')
print('─' * 70)
results_all = knn_search(client, embed(query), k=3)
for r in results_all:
    print(f"  [{r['score']:.4f}] {r['episode_title']}")
    print(f"           {r['chunk_text'][:150]}...")
    print()

UNFILTERED k-NN search:
──────────────────────────────────────────────────────────────────────
  [0.6558] Yury Malkov - Staff Engineer, Twitter - Author of the most adopted ANN algorithm HNSW
           that that that it will incur because basically as you build the H and SW you also use memory quite a bit right so I wanted to hear your opinion on tha...

  [0.6466] Filip Haltmayer (Data Engineer, Ziliz) on Milvus vector database and working with clients
           you can actually load them in memory. And then like from there, they build the, so for the clusters, they build the H&SW, the graph kind of layout, ri...

  [0.6429] Jo Bergum - Distinguished Engineer, Yahoo! Vespa - Journey of Vespa from Sparse into Neural Search
           then you can compute the overlap between those two and that's typically then what's used in the recall at k right so I did two blog posts on what I ca...



In [5]:
# Pick an episode title from the list above
target_episode = episodes[0]['key']  # first episode alphabetically
print(f'FILTERED to: "{target_episode}"')
print('─' * 70)

results_filtered = knn_search_filtered(
    client,
    embed(query),
    k=5,
    episode_title=target_episode,
)

if not results_filtered:
    print('  No results: this episode may not discuss this topic.')
else:
    for r in results_filtered:
        print(f"  [{r['score']:.4f}] {r['episode_title']}")
        print(f"           {r['chunk_text'][:150]}...")
        print()

FILTERED to: "Adding ML layer to Search: Hybrid Search Optimizer with Daniel Wrigley and Eric Pugh"
──────────────────────────────────────────────────────────────────────
  No results: this episode may not discuss this topic.


## 3 · Filter by Date Range

The `pub_date` field stores RFC 2822 date strings (e.g. `"Mon, 15 Jan 2024 12:00:00 GMT"`).
We can use OpenSearch `range` queries to scope to a time window.

> **Note:** Since `pub_date` is stored as `keyword` (not `date` type), range filtering
> does lexicographic comparison. This works for consistently-formatted date strings.
> For production, consider indexing dates as the `date` type.

In [6]:
query = "What are the latest trends in vector search?"

# Get the date strings to use for filtering
all_dates = sorted([d['key'] for d in resp['aggregations']['dates']['buckets']])
mid_date = all_dates[len(all_dates)//2] if all_dates else None

print(f'Filtering to episodes published after: {mid_date}')
print('─' * 70)

results_recent = knn_search_filtered(
    client,
    embed(query),
    k=5,
    pub_date_gte=mid_date,
)

for r in results_recent:
    print(f"  [{r['score']:.4f}] {r['pub_date'][:20]:20s}  {r['episode_title']}")
    print(f"           {r['chunk_text'][:120]}...")
    print()

Filtering to episodes published after: Thu, 09 Jun 2022 14:51:07 GMT
──────────────────────────────────────────────────────────────────────
  [0.7541] Wed, 16 Feb 2022 16:  Amin Ahmad - CTO, Vectara - Algolia / Elasticsearch-like search product on neural search principles
           people start to understand how useful vector embeddings can be, and that's established, that you you'll have a, you know...

  [0.7143] Tue, 12 Apr 2022 12:  Jo Bergum - Distinguished Engineer, Yahoo! Vespa - Journey of Vespa from Sparse into Neural Search
           you've mentioned vector search I'm actually curious like when in Westbar journey you know did you first hear about vecto...



## 4 · Combined: Episode + Date Filters

Filters compose with `bool.must`. All conditions must match.

In [7]:
query = "embeddings and search quality"

results_combo = knn_search_filtered(
    client,
    embed(query),
    k=5,
    episode_title=target_episode,
    pub_date_gte=all_dates[0] if all_dates else None,
)

print(f'Combined filter: episode="{target_episode}", date >= "{all_dates[0][:20]}..."')
print(f'Results: {len(results_combo)}')
print('─' * 70)
for r in results_combo:
    print(f"  [{r['score']:.4f}] {r['episode_title'][:50]}")
    print(f"           {r['chunk_text'][:150]}...")
    print()

Combined filter: episode="Adding ML layer to Search: Hybrid Search Optimizer with Daniel Wrigley and Eric Pugh", date >= "Fri, 07 Nov 2025 05:..."
Results: 0
──────────────────────────────────────────────────────────────────────


## 5 · Pre-filter vs Post-filter

OpenSearch supports two filtering strategies with k-NN:

| Strategy | How it works | Trade-off |
|---|---|---|
| **Post-filter** (what we used) | Run k-NN on full index, then filter results | May return < k results if many are filtered out |
| **Pre-filter** (Lucene engine) | Filter first, then run k-NN on the subset | Always returns k results, but requires `engine: lucene` |

Our index uses `engine: nmslib` with HNSW, which supports **post-filtering**.
For pre-filtering, you'd need to create the index with `engine: lucene`.

**When to use which:**
- **Post-filter**: when your filter is not too selective (keeps >10% of docs)
- **Pre-filter**: when your filter is very selective (e.g., single user, single episode)

To compensate for post-filter dropping results, retrieve more candidates:
```python
# Retrieve 5x candidates, post-filter, then take top-5
results = knn_search_filtered(client, embed(q), k=25, episode_title=...)
results = results[:5]
```

In [8]:
# Over-retrieve then trim - A practical pattern for post-filtering
query = "What is approximate nearest neighbour search?"
OVER_FETCH = 25
FINAL_K = 5

results = knn_search_filtered(
    client,
    embed(query),
    k=OVER_FETCH,
    episode_title=target_episode,
)
results = results[:FINAL_K]

print(f'Over-fetched {OVER_FETCH}, filtered to episode, kept top {FINAL_K}:')
for r in results:
    print(f"  [{r['score']:.4f}] {r['episode_title'][:50]}")

Over-fetched 25, filtered to episode, kept top 5:


## Key Takeaways

- **Metadata filtering** lets you scope vector search to specific episodes, date ranges, or any structured field
- OpenSearch uses **post-filtering** with nmslib/HNSW, retrieve first, filter after
- Over-fetch candidates (3–5x) when using selective filters to compensate for post-filter drop-off
- For very selective filters, consider switching to `engine: lucene` which supports **pre-filtering**
- Combine with re-ranking (Notebook 03) for the best precision